get_emebbedding.py

In [ ]:
import gguf
import numpy as np
import csv

# Byte-Level BPE tokenlerini okunabilir hale çevirmek için dönüşüm haritası hazırlama
def _get_byte_decoder():
    bs = (
        list(range(ord("!"), ord("~") + 1)) +
        list(range(ord("¡"), ord("¬") + 1)) +
        list(range(ord("®"), ord("ÿ") + 1))
    )
    cs = bs[:]
    n = 0
    for b in range(2**8):
        if b not in bs:
            bs.append(b)
            cs.append(2**8 + n)
            n += 1
    return {chr(c): b for c, b in zip(cs, bs)}

bpe_decoder_map = _get_byte_decoder()

def clean_bpe_token(token_utf8_str):
    "BPE sembollerini ham byte formatına alıp tekrar UTF-8 olarak okunaklı metne çevirir."
    try:
        # BPE stringindeki her karakteri orjinal byte değerine dönüştürüyoruz
        raw_bytes = bytes([bpe_decoder_map.get(c, ord(c)) for c in token_utf8_str])
        # Şimdi bunu normal bir UTF-8 metni olarak çözümlüyoruz (Bozuksa kaçış sekansına izin veriyoruz)
        return raw_bytes.decode('utf-8', errors='backslashreplace')
    except Exception:
        return token_utf8_str

def read_gguf_tokens(model_path, output_csv="tokens.csv"):
    print(f"Reading model from: {model_path}")
    
    reader = gguf.GGUFReader(model_path)
    token_field_key = "tokenizer.ggml.tokens"
    
    if token_field_key in reader.fields:
        field = reader.fields[token_field_key]
        parts = field.parts  
        
        try:
            total_tokens = parts[4][0]
        except:
            total_tokens = "Bilinmiyor"
            
        token_byte_arrays = parts[6::2]
        
        print(f"{'='*40}")
        print(f"Modelde toplam {total_tokens} adet token bulundu.")
        print(f"Tokenler '{output_csv}' dosyasına kaydediliyor...")
        print(f"{'='*40}")
        
        with open(output_csv, 'w', encoding='utf-8-sig', newline='') as f:
            writer = csv.writer(f)
            writer.writerow(["ID", "Token"])
            
            for i, token_array in enumerate(token_byte_arrays):
                token_bytes = bytes(token_array)
                # Önce standart utf-8 string olarak alıyoruz (ör. Ġis)
                bpe_str = token_bytes.decode('utf-8', errors='backslashreplace')
                # Sonra bunu tersine çevirerek normal (okunaklı) bir string'e çeviriyoruz
                human_readable_str = clean_bpe_token(bpe_str)
                writer.writerow([i, human_readable_str])
                
        print(f"\nİşlem bitti! Tüm tokenler '{output_csv}' dosyasına yazıldı.")
            
    else:
        print(f"Hata: GGUF dosyasında '{token_field_key}' bulunamadı.")
        for key in reader.fields.keys():
            if "token" in key.lower():
                print(f"- {key}")

def read_gguf_weights(model_path):
    # İhtiyaç olursa diye burada ağırlık okuma fonksiyonu
    reader = gguf.GGUFReader(model_path)
    for tensor in reader.tensors:
        print(tensor.name, tensor.shape)

if __name__ == '__main__':
    # GGUF dosyanızın yolu
    model_dosya_yolu = "./model_weight/Turkish-Gemma-9b-T1.Q4_K_S.gguf" 
    
    # Bütün tokenleri okuyup tokens.csv dosyasına yazdırmak için
    read_gguf_tokens(model_dosya_yolu, output_csv="tokens.csv")


get_weight.py

In [ ]:
import gguf
import numpy as np
import csv
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Byte-Level BPE tokenlerini okunabilir hale çevirmek için dönüşüm haritası hazırlama
def _get_byte_decoder():
    bs = (
        list(range(ord("!"), ord("~") + 1)) +
        list(range(ord("¡"), ord("¬") + 1)) +
        list(range(ord("®"), ord("ÿ") + 1))
    )
    cs = bs[:]
    n = 0
    for b in range(2**8):
        if b not in bs:
            bs.append(b)
            cs.append(2**8 + n)
            n += 1
    return {chr(c): b for c, b in zip(cs, bs)}

bpe_decoder_map = _get_byte_decoder()

def clean_bpe_token(token_utf8_str):
    "BPE sembollerini ham byte formatına alıp tekrar UTF-8 olarak okunaklı metne çevirir."
    try:
        raw_bytes = bytes([bpe_decoder_map.get(c, ord(c)) for c in token_utf8_str])
        return raw_bytes.decode('utf-8', errors='backslashreplace')
    except Exception:
        return token_utf8_str

def read_gguf_tokens(model_path, output_csv="tokens.csv"):
    print(f"Reading model from: {model_path}")
    
    reader = gguf.GGUFReader(model_path)
    token_field_key = "tokenizer.ggml.tokens"
    
    if token_field_key in reader.fields:
        field = reader.fields[token_field_key]
        parts = field.parts  
        
        try:
            total_tokens = parts[4][0]
        except:
            total_tokens = "Bilinmiyor"
            
        token_byte_arrays = parts[6::2]
        
        print(f"{'='*40}")
        print(f"Modelde toplam {total_tokens} adet token bulundu.")
        print(f"Tokenler '{output_csv}' dosyasına kaydediliyor...")
        print(f"{'='*40}")
        
        with open(output_csv, 'w', encoding='utf-8-sig', newline='') as f:
            writer = csv.writer(f)
            writer.writerow(["ID", "Token"])
            
            for i, token_array in enumerate(token_byte_arrays):
                token_bytes = bytes(token_array)
                bpe_str = token_bytes.decode('utf-8', errors='backslashreplace')
                human_readable_str = clean_bpe_token(bpe_str)
                writer.writerow([i, human_readable_str])
                
        print(f"\nİşlem bitti! Tüm tokenler '{output_csv}' dosyasına yazıldı.")
            
    else:
        print(f"Hata: GGUF dosyasında '{token_field_key}' bulunamadı.")
        for key in reader.fields.keys():
            if "token" in key.lower():
                print(f"- {key}")

def plot_gguf_tensor_heatmap(model_path, target_tensor_name="token_embd.weight", max_size=100):
    """Tek bir tensorün ısı haritasını ekranda gösterir."""
    print(f"Ağırlıklar okunuyor: {model_path}")
    reader = gguf.GGUFReader(model_path)
    
    target_tensor = None
    for tensor in reader.tensors:
        if tensor.name == target_tensor_name:
            target_tensor = tensor
            break
            
    if target_tensor is None:
        print(f"Hata: '{target_tensor_name}' adlı tensor bulunamadı.")
        return
        
    print(f"Tensor bulundu: {target_tensor.name}, Shape: {target_tensor.shape}")
    
    data = np.array(target_tensor.data).flatten()
    num_elements = len(data)
    target_elements = max_size * max_size
    
    if num_elements >= target_elements:
        matrix_slice = np.array(data[:target_elements], dtype=np.float32).reshape(max_size, max_size)
    else:
        side = int(np.sqrt(num_elements))
        if side == 0:
            matrix_slice = np.array(data, dtype=np.float32).reshape(1, num_elements)
        else:
            matrix_slice = np.array(data[:side * side], dtype=np.float32).reshape(side, side)
            
    plt.figure(figsize=(10, 8))
    sns.heatmap(matrix_slice, cmap="coolwarm", cbar_kws={"label": "Ağırlık Değeri Özeti"})
    
    plt.title(f"Ağırlık Matrisi Isı Haritası\nTensor: {target_tensor.name} (Kesit: {matrix_slice.shape})", fontsize=14)
    plt.xlabel("Sütunlar", fontsize=12)
    plt.ylabel("Satırlar", fontsize=12)
    plt.tight_layout()
    plt.show()

def save_all_tensor_heatmaps(model_path, output_dir="tensor_heatmaps", max_size=100, max_tensors=None):
    """GGUF modelindeki tüm tensorlerin ısı haritalarını 'output_dir' klasörüne kaydeder."""
    print(f"\nAğırlıklar okunuyor: {model_path}")
    reader = gguf.GGUFReader(model_path)
    
    # Çıktı klasörünü oluştur (varsa hata vermez)
    os.makedirs(output_dir, exist_ok=True)
    
    tensors = reader.tensors
    if max_tensors is not None:
        tensors = tensors[:max_tensors]
        
    print(f"Toplam {len(tensors)} tensor görselleştirilip '{output_dir}' klasörüne kaydedilecek...\n")
    
    for i, tensor in enumerate(tensors):
        print(f"[{i+1}/{len(tensors)}] İşleniyor: {tensor.name} (Shape: {tensor.shape})")
        
        data = np.array(tensor.data).flatten()
        num_elements = len(data)
        
        if num_elements == 0:
            print(f"  {tensor.name} boş olduğu için atlanıyor.")
            continue
            
        target_elements = max_size * max_size
        
        if num_elements >= target_elements:
            matrix_slice = np.array(data[:target_elements], dtype=np.float32).reshape(max_size, max_size)
        else:
            side = int(np.sqrt(num_elements))
            if side == 0:
                matrix_slice = np.array(data, dtype=np.float32).reshape(1, len(data))
            else:
                matrix_slice = np.array(data[:side * side], dtype=np.float32).reshape(side, side)
                
        plt.figure(figsize=(10, 8))
        sns.heatmap(matrix_slice, cmap="coolwarm", cbar_kws={"label": "Ağırlık Değeri Özeti"})
        
        plt.title(f"Ağırlık Matrisi Isı Haritası\nTensor: {tensor.name} (Kesit: {matrix_slice.shape})", fontsize=14)
        plt.xlabel("Sütunlar", fontsize=12)
        plt.ylabel("Satırlar", fontsize=12)
        plt.tight_layout()
        
        # Tensor adında dosya sistemini bozacak semboller (/, \) varsa '_' ile değiştir
        safe_name = tensor.name.replace('/', '_').replace('\\', '_')
        # Örnek Dosya Adı: 001_token_embd.weight.png
        file_path = os.path.join(output_dir, f"{i+1:03d}_{safe_name}.png")
        
        # Çıktıyı dosyaya kaydet ve belleği temizlemek için şekli kapat
        plt.savefig(file_path, dpi=100)
        plt.close()
        
    print(f"\nİşlem tamamlandı! Bütün ısı haritaları '{output_dir}' klasörüne kaydedildi.")

if __name__ == '__main__':
    # GGUF dosyanızın yolu
    model_dosya_yolu = "./model_weight/Turkish-Gemma-9b-T1.Q4_K_S.gguf" 
    
    # 1) Token okuma işlemini yapabilirsiniz
    # read_gguf_tokens(model_dosya_yolu, output_csv="tokens.csv")
    
    # 2) Sadece tek bir tensor çıkarıp ekranda görmek için
    # plot_gguf_tensor_heatmap(model_dosya_yolu, target_tensor_name="token_embd.weight", max_size=100)
    
    # 3) BÜTÜN tensorlerin ısı haritalarını sırasıyla bir klasöre kaydetmek için
    # Not: Modelde çok tensor varsa hepsini kaydetmek biraz zaman alabilir.
    # max_tensors=20 diyerek sadece ilk 20 tensorü almasını sağlayabilirsiniz. (Sınır koymak istemiyorsanız None yapın)
    save_all_tensor_heatmaps(model_dosya_yolu, output_dir="tensor_heatmaps", max_size=100, max_tensors=None)
